# Day 33: Advanced RAG - Cross-Encoder Reranking

Welcome to Day 33 of your AI Engineering journey!

## Core Theory (Just-in-Time)

**Why Reranking?**
Standard Vector Search (Bi-Encoders) is fast and excellent for initial retrieval across millions of documents. However, it relies on comparing single embeddings independently, missing nuanced relationships between the query and the document.

**The Cross-Encoder Solution**
A Cross-Encoder passes both the user query and the retrieved document simultaneously into a Transformer model. This allows the model's attention mechanism to directly compare the query and document tokens, producing a highly accurate relevance score. Since Cross-Encoders are computationally expensive, we use a two-stage approach:
1.  **Stage 1 (Retrieval):** Use a Bi-Encoder (like standard embeddings in Qdrant) to quickly retrieve the top 10-50 candidates.
2.  **Stage 2 (Reranking):** Pass those top candidates through a Cross-Encoder to re-order them and select the absolute best (e.g., top 3-5) for the LLM.

**Common Production Pitfalls:**
- **Over-Reranking:** Passing hundreds of documents to a Cross-Encoder will cause massive latency spikes. Always keep the initial retrieval pool small (k=10 to 50).
- **Ignoring Score Thresholds:** Cross-Encoders give absolute scores. If the highest score is still very low, your system should probably fallback or say "I don't know" rather than feeding bad context to the LLM.

## Code Implementation

We will implement a two-stage retrieval pipeline using Qdrant as the vector store and LangChain's `ContextualCompressionRetriever` paired with a HuggingFace Cross-Encoder.

In [1]:
from typing import List
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
import uuid

# 1. Setup Models
print("Loading Bi-Encoder for retrieval...")
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

print("Loading Cross-Encoder for reranking...")
cross_encoder_model = HuggingFaceCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")

# 2. Setup Vector Store (Using direct QdrantClient for version compatibility)
client = QdrantClient(":memory:")
collection_name = "demo_reranking"

client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE)
)

documents = [
    Document(page_content="Apples are grown in Washington state and are red or green."),
    Document(page_content="Apple Inc. announced a new iPhone today."),
    Document(page_content="A healthy diet includes fruits like apples and oranges."),
    Document(page_content="The company Apple was founded by Steve Jobs."),
    Document(page_content="To bake an apple pie, you need cinnamon and sugar.")
]

# Embed and insert manually
points = []
for doc in documents:
    vec = embeddings.embed_query(doc.page_content)
    points.append(
        PointStruct(
            id=str(uuid.uuid4()),
            vector=vec,
            payload={"page_content": doc.page_content}
        )
    )

client.upsert(collection_name=collection_name, points=points)

# 3. Create the Retriver Pipeline (Custom logic for Qdrant v1.19.0+)
# Note: In production we often wrap this in a BaseRetriever, but here we show the raw execution flow.

def search_and_rerank(query: str) -> List[Document]:
    """
    Executes a two-stage retrieval process: initial vector search followed by Cross-Encoder reranking.
    
    Args:
        query: The user's search query.
        
    Returns:
        A list of the top N reranked Documents.
    """
    print(f"\nQuery: '{query}'")
    
    # Stage 1: Fast Base Retrieval (Bi-Encoder)
    query_vec = embeddings.embed_query(query)
    search_result = client.query_points(
        collection_name=collection_name,
        query=query_vec,
        limit=4
    )
    
    base_docs = [Document(page_content=point.payload["page_content"]) for point in search_result.points]
    
    print("\n--- Stage 1: Base Retrieval (Bi-Encoder) ---")
    for i, doc in enumerate(base_docs):
        print(f"{i+1}. {doc.page_content}")

    # Stage 2: Rerank the top 4 and keep the best 2
    compressor = CrossEncoderReranker(model=cross_encoder_model, top_n=2)
    reranked_docs = compressor.compress_documents(base_docs, query)
    
    print("\n--- Stage 2: Reranked Retrieval (Cross-Encoder) ---")
    for i, doc in enumerate(reranked_docs):
        # Note: compressor adds a 'relevance_score' to the metadata
        score = doc.metadata.get('relevance_score', 'N/A')
        print(f"{i+1}. [Score: {score}] {doc.page_content}")
        
    return reranked_docs

search_and_rerank("Tell me about the technology company.")


Loading Bi-Encoder for retrieval...


/app/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:  80%|███████▉  | 159/199 [00:00<00:00, 1577.91it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1756.09it/s]

Loading Cross-Encoder for reranking...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1973.37it/s]


Query: 'Tell me about the technology company.'

--- Stage 1: Base Retrieval (Bi-Encoder) ---
1. The company Apple was founded by Steve Jobs.
2. Apple Inc. announced a new iPhone today.
3. Apples are grown in Washington state and are red or green.
4. A healthy diet includes fruits like apples and oranges.

--- Stage 2: Reranked Retrieval (Cross-Encoder) ---
1. [Score: N/A] The company Apple was founded by Steve Jobs.
2. [Score: N/A] Apple Inc. announced a new iPhone today.


[Document(page_content='The company Apple was founded by Steve Jobs.'),
 Document(page_content='Apple Inc. announced a new iPhone today.')]

## Practical Lab / Homework

**Task:** Expand the reranking system to handle an edge case: Score Thresholds.

Currently, the `CrossEncoderReranker` always returns the `top_n` documents, even if they are completely irrelevant to the query. 

Your job is to:
1. Add 5 more random documents to the vector store about completely unrelated topics (e.g., cars, space).
2. Create a custom function that intercepts the output of the reranker.
3. Inspect the reranker scores (they are usually stored in `doc.metadata['relevance_score']`).
4. Filter out any documents where the score is below a certain threshold (e.g., 0.0), returning only highly relevant documents.
5. Test it with a query like "How do rockets work?" to ensure it returns nothing instead of returning Apple-related docs.

In [2]:
def search_and_rerank_with_threshold(query: str, threshold: float = 0.0) -> List[Document]:
    """
    Executes a two-stage retrieval process with a strict score threshold.
    """
    query_vec = embeddings.embed_query(query)
    search_result = client.query_points(
        collection_name=collection_name,
        query=query_vec,
        limit=10
    )
    
    base_docs = [Document(page_content=point.payload["page_content"]) for point in search_result.points]
    
    compressor = CrossEncoderReranker(model=cross_encoder_model, top_n=5)
    reranked_docs = compressor.compress_documents(base_docs, query)
    
    filtered_docs = [doc for doc in reranked_docs if doc.metadata.get('relevance_score', -999) > threshold]
    
    print(f"\nQuery: '{query}'")
    print(f"--- Filtered Results (Threshold > {threshold}) ---")
    for i, doc in enumerate(filtered_docs):
        print(f"{i+1}. [Score: {doc.metadata.get('relevance_score')}] {doc.page_content}")
        
    return filtered_docs

# Insert unrelated documents
unrelated_docs = [
    Document(page_content="The Falcon Heavy rocket launched successfully."),
    Document(page_content="Electric cars are becoming more popular."),
    Document(page_content="SpaceX is a company that builds rockets."),
    Document(page_content="Ford announced a new electric Mustang."),
    Document(page_content="Astronauts travel to the International Space Station.")
]

points = []
for doc in unrelated_docs:
    vec = embeddings.embed_query(doc.page_content)
    points.append(
        PointStruct(
            id=str(uuid.uuid4()),
            vector=vec,
            payload={"page_content": doc.page_content}
        )
    )
client.upsert(collection_name=collection_name, points=points)

search_and_rerank_with_threshold("How do rockets work?")



Query: 'How do rockets work?'
--- Filtered Results (Threshold > 0.0) ---


[]